# Ridge model on surprise features

**Competition metric.** ΔR² = R²(`target_pct` ~ prediction + `earnings_surprise`) − R²(`target_pct` ~ `earnings_surprise`), both OLS with an intercept. A prediction scores only for what it explains beyond the EPS surprise.

**Plan.**
1. Missing values: what is missing and how it is handled.
2. Reference scores on train: single features and the competition baselines.
3. Ridge hyperparameters by time-ordered cross-validation on train (alpha, feature set, target mode, log transform), scored by ΔR².
4. Fit the chosen configuration on all train events.
5. Evaluate once on test with ΔR², against the competition baselines and the plain `net_w` score, with paired bootstrap comparisons.

Data: `data/surprise/features_v2.csv` (1,076 train events, 3–13 Aug 2026; 289 test events, 13 Aug–22 Sep).

In [1]:
%load_ext autoreload
%autoreload 2

import json

import numpy as np
import pandas as pd

from ml import metrics, model, preprocess
from suprise_dim import data, features

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

feat = pd.read_csv(features.features_path("v2"))
train = feat[feat["split"] == "train"].reset_index(drop=True)
test = feat[feat["split"] == "test"].reset_index(drop=True)
BASELINES = sorted(c for c in feat.columns if c.startswith("baseline__"))
print(f"train {len(train)} | test {len(test)} | baselines: {[b.split('__')[1] for b in BASELINES]}")

train 1076 | test 289 | baselines: ['deepseek', 'gemini', 'moonshotai', 'openai', 'z-ai']


## 1. Missing values

- **Surprise features** are complete: a dimension with no surprise scores 0, and the 18 test events whose call facts have no content are all zeros with `no_surprises = 1`.
- **Option statistics** are missing for most events. Inside the model they are imputed with the training-fold median plus a missing-value indicator (`preprocess.build_preprocessor`), fitted on training rows only.
- **`earnings_surprise`** is missing where the archive has no EPS. It is not a model input. In the metric, rows without it are dropped (the count is reported). The `residual` target mode imputes the training median to residualize.
- **Baselines** are missing for some events, so comparisons use rows where every prediction exists.

In [2]:
cols = preprocess.OPTION_COLUMNS + ["earnings_surprise"] + BASELINES
preprocess.missing_summary(feat, cols)

split,test,train,all
opt_implied_earnings_volatility,0.536,0.586,0.575
opt_implied_absolute_earnings_move,0.536,0.586,0.575
opt_skew_25_delta,0.626,0.767,0.737
earnings_surprise,0.062,0.045,0.048
baseline__deepseek,0.024,0.017,0.018
baseline__gemini,0.007,0.000,0.001
baseline__moonshotai,0.010,0.042,0.035
baseline__z-ai,0.017,0.008,0.010


## 2. Reference scores on train

Single features turned into percentiles by rank, and the competition baselines. These show the scale of ΔR² before any model is fitted.

In [3]:
def references(frame):
    refs = {name: model.rank_to_percentile(frame[name]) for name in ["net_w", "net_w_first3", "net"]}
    refs |= {b.split("__")[1]: frame[b].to_numpy() for b in BASELINES}
    return refs

metrics.evaluate(train["target_pct"], references(train), train["earnings_surprise"]).round(4)

,n,r2_base,r2_full,delta_r2,spearman
net_w_first3,1028.0,0.0035,0.2260,0.2225,0.4658
net_w,1028.0,0.0035,0.2256,0.2221,0.4676
net,1028.0,0.0035,0.2019,0.1985,0.4427
gemini,1028.0,0.0035,0.1758,0.1724,0.3984
z-ai,1019.0,0.0035,0.1555,0.1520,0.3734
deepseek,1011.0,0.0034,0.1520,0.1485,0.3704
moonshotai,985.0,0.0035,0.1380,0.1345,0.3541
openai,1028.0,0.0035,0.1100,0.1065,0.3201


## 3. Cross-validation on train

Expanding-window folds in time order (`model.time_folds`, 5 folds; fold k trains on the earliest blocks and validates on the next). Each configuration is scored by ΔR² on every validation fold (mean and sd) and on the pooled out-of-fold predictions.

Grid:
- `feature_set`: `totals`, `net` (per-dimension net), `posneg` (per-dimension positive and negative), `all`
- `alpha`: 0.1 to 10,000 on a log scale
- `target_mode`: `raw` (fit `target_pct`) or `residual` (fit what `earnings_surprise` leaves unexplained)
- `log_counts`: signed log of count features, or not

In [4]:
sets = preprocess.feature_sets(feat.columns)
print({name: len(cols) for name, cols in sets.items()})

alphas = np.logspace(-1, 4, 11)
grid = model.grid_search(train, sets, alphas)
grid.drop(columns="fold_delta_r2").head(15).round(4)

{'totals': 10, 'net': 22, 'posneg': 37, 'all': 59}


,feature_set,n_features,alpha,target_mode,log_counts,cv_delta_r2_mean,cv_delta_r2_sd,oof_delta_r2,oof_spearman
0,all,59,10000.0000,raw,False,0.2680,0.0867,0.2220,0.4839
1,all,59,10000.0000,residual,False,0.2679,0.0865,0.2250,0.4882
2,all,59,3162.2777,raw,False,0.2649,0.0842,0.2384,0.4921
3,all,59,3162.2777,residual,False,0.2647,0.0838,0.2394,0.4933
4,all,59,10000.0000,raw,True,0.2608,0.0846,0.2164,0.4714
5,all,59,10000.0000,residual,True,0.2606,0.0842,0.2195,0.4768
6,all,59,3162.2777,raw,True,0.2577,0.0826,0.2324,0.4803
7,all,59,3162.2777,residual,True,0.2574,0.0819,0.2334,0.4824
8,all,59,1000.0000,raw,False,0.2566,0.0807,0.2440,0.4906
9,all,59,1000.0000,residual,False,0.2563,0.0795,0.2447,0.4911


In [5]:
# Best configuration per feature set and per target mode, to see what drives the score.
summary = grid.drop(columns="fold_delta_r2")
display(summary.groupby("feature_set").head(1).round(4))
summary.groupby("target_mode").head(1).round(4)

,feature_set,n_features,alpha,target_mode,log_counts,cv_delta_r2_mean,cv_delta_r2_sd,oof_delta_r2,oof_spearman
0,all,59,10000.0,raw,False,0.2680,0.0867,0.2220,0.4839
10,totals,10,10000.0,raw,False,0.2529,0.0825,0.1845,0.4558
11,posneg,37,10000.0,raw,False,0.2524,0.0834,0.1957,0.4564
17,net,22,10000.0,raw,False,0.2509,0.0775,0.1865,0.4498


,feature_set,n_features,alpha,target_mode,log_counts,cv_delta_r2_mean,cv_delta_r2_sd,oof_delta_r2,oof_spearman
0,all,59,10000.0,raw,False,0.2680,0.0867,0.222,0.4839
1,all,59,10000.0,residual,False,0.2679,0.0865,0.225,0.4882


### Chosen configuration

The top row by mean fold ΔR². With ~200 configurations on 1,076 events the winner's CV score is optimistic, so the test score below is the number to trust.

In [6]:
best = grid.iloc[0]
print(best.drop("fold_delta_r2").to_dict())
print("fold ΔR²:", np.round(best["fold_delta_r2"], 4))

{'feature_set': 'all', 'n_features': 59, 'alpha': 10000.0, 'target_mode': 'raw', 'log_counts': False, 'cv_delta_r2_mean': 0.26795838462338567, 'cv_delta_r2_sd': 0.08671207389806776, 'oof_delta_r2': 0.22201134615557427, 'oof_spearman': 0.4838829439313566}
fold ΔR²: [0.1828 0.3482 0.2529 0.3673 0.1886]


## 4. Fit on all train events

In [7]:
final = model.fit(train, sets[best["feature_set"]], best["alpha"],
                  target_mode=best["target_mode"], log_counts=bool(best["log_counts"]))
train_pred = final.predict(train)
print("train (in-sample) ΔR²:", round(metrics.delta_r2(train["target_pct"], train_pred, train["earnings_surprise"])["delta_r2"], 4))
final.coefficients().head(20).round(4).to_frame("coef")

train (in-sample) ΔR²: 0.2509


,coef
net_w_first3,0.0079
net_w,0.0075
net_first3,0.0074
net,0.0069
net_per_item,0.0068
net_w__forward_guidance,0.0063
neg_w__forward_guidance,-0.0060
n_pos,0.0060
n_neg,-0.0059
net_w__operating_profitability,0.0051


## 5. Test evaluation

Scored once. All predictions are compared on the same rows: test events with `earnings_surprise` and every baseline present.

In [8]:
test_pred = final.predict(test)
preds = {"ridge": test_pred} | references(test)

common = test[["earnings_surprise"] + BASELINES].notna().all(axis=1).to_numpy()
print(f"test rows: {len(test)} | with earnings_surprise and all baselines: {common.sum()}")
metrics.evaluate(test["target_pct"][common], {k: v[common] for k, v in preds.items()}, test["earnings_surprise"][common]).round(4)

test rows: 289 | with earnings_surprise and all baselines: 256


,n,r2_base,r2_full,delta_r2,spearman
net_w_first3,256.0,0.0004,0.2155,0.2151,0.4680
ridge,256.0,0.0004,0.1996,0.1992,0.4455
net_w,256.0,0.0004,0.1665,0.1661,0.4086
net,256.0,0.0004,0.1617,0.1613,0.4030
gemini,256.0,0.0004,0.1057,0.1053,0.3005
z-ai,256.0,0.0004,0.0913,0.0909,0.3022
moonshotai,256.0,0.0004,0.0892,0.0888,0.2939
openai,256.0,0.0004,0.0826,0.0822,0.2741
deepseek,256.0,0.0004,0.0687,0.0683,0.2811


In [9]:
# Each prediction on every test row it can be scored on (n differs by prediction).
metrics.evaluate(test["target_pct"], preds, test["earnings_surprise"]).round(4)

,n,r2_base,r2_full,delta_r2,spearman
net_w_first3,271.0,0.0002,0.1938,0.1935,0.4171
ridge,271.0,0.0002,0.1853,0.1851,0.4113
net_w,271.0,0.0002,0.1515,0.1513,0.3811
net,271.0,0.0002,0.1501,0.1499,0.3838
gemini,270.0,0.0003,0.0983,0.0981,0.2816
z-ai,266.0,0.0006,0.0971,0.0966,0.3039
moonshotai,268.0,0.0002,0.0745,0.0743,0.2631
deepseek,264.0,0.0002,0.0716,0.0714,0.2796
openai,271.0,0.0002,0.0637,0.0634,0.2397


In [10]:
# Paired bootstrap on the common rows: ridge minus each other prediction.
samples = metrics.paired_bootstrap(test["target_pct"][common], {k: v[common] for k, v in preds.items()},
                                   test["earnings_surprise"][common], n_boot=2000, seed=0)
print("ridge ΔR² 95% interval:", samples["ridge"].quantile([0.025, 0.975]).round(4).tolist())
metrics.compare_to(samples, "ridge").round(4)

ridge ΔR² 95% interval: [0.115, 0.2898]


,mean_diff,ci_low,ci_high,P(ridge better)
net_w,0.0327,0.0068,0.0571,0.9880
net_w_first3,-0.0159,-0.0625,0.0271,0.2460
net,0.0371,0.0076,0.0658,0.9935
deepseek,0.1292,0.0634,0.1994,1.0000
gemini,0.0930,0.0298,0.1583,0.9960
moonshotai,0.1096,0.0419,0.1807,0.9985
openai,0.1153,0.0467,0.1889,0.9985
z-ai,0.1065,0.0334,0.1835,0.9975


## 6. Save test predictions and the chosen configuration

In [11]:
out = test[["event_id", "event_datetime", "target_pct", "earnings_surprise"]].assign(ridge=test_pred)
out.to_csv(data.OUT_DIR / "ridge_v2_test_predictions.csv", index=False)

config = {k: (v.item() if hasattr(v, "item") else v) for k, v in best.drop("fold_delta_r2").to_dict().items()}
config["features"] = final.features
(data.OUT_DIR / "ridge_v2_config.json").write_text(json.dumps(config, indent=2))
print(config | {"features": f"{len(final.features)} columns"})

{'feature_set': 'all', 'n_features': 59, 'alpha': 10000.0, 'target_mode': 'raw', 'log_counts': False, 'cv_delta_r2_mean': 0.26795838462338567, 'cv_delta_r2_sd': 0.08671207389806776, 'oof_delta_r2': 0.22201134615557427, 'oof_spearman': 0.4838829439313566, 'features': '59 columns'}
